In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/candidate/__results__.html
/kaggle/input/candidate/__notebook__.ipynb
/kaggle/input/candidate/__output__.json
/kaggle/input/candidate/custom.css
/kaggle/input/candidate/candidates/user_candidates.pkl
/kaggle/input/candidate/candidates/user_segments.pkl
/kaggle/input/fe-tiep/__results__.html
/kaggle/input/fe-tiep/__notebook__.ipynb
/kaggle/input/fe-tiep/__output__.json
/kaggle/input/fe-tiep/custom.css
/kaggle/input/fe-tiep/feature_engineering/content_similarity.pkl
/kaggle/input/fe-tiep/feature_engineering/validation_data.parquet
/kaggle/input/fe-tiep/feature_engineering/cold_start_maps.pkl
/kaggle/input/fe-tiep/feature_engineering/customer_features.parquet
/kaggle/input/fe-tiep/feature_engineering/item_similarity_cosine.pkl
/kaggle/input/fe-tiep/feature_engineering/interaction_features.parquet
/kaggle/input/fe-tiep/feature_engineering/user_purchase_history.pkl
/kaggle/input/fe-tiep/feature_engineering/test_ground_truth.parquet
/kaggle/input/fe-tiep/feature_engineering/coo

In [2]:
# =============================================================================
# NOTEBOOK 5: RANKING MODEL & DUAL EVALUATION (FINAL COMPLETE)
# =============================================================================
# STRATEGY:
# 1. Train Model: Chỉ train trên dữ liệu của Warm Users.
# 2. Inference: 
#    - Warm Users -> Dùng Model Rerank.
#    - Cold Users -> Dùng Candidate list (đã sort ở NB4).
#    - Ghost Users (Missing in Candidates) -> Dùng Global Best Sellers (Safety Net).
# 3. Evaluation: 
#    - Case 1: No Cold Start (Mean metrics over Warm Users).
#    - Case 2: With Cold Start (Mean metrics over All Test Users).
# =============================================================================

import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
import pickle
import os
import gc
import json
import random
import math
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
FEATURE_PATH = "/kaggle/input/fe-tiep/feature_engineering"
CANDIDATE_PATH = "/kaggle/input/candidate/candidates"
OUTPUT_PATH = "/kaggle/working/ranking"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Ranking Config
NEG_SAMPLE_RATIO = 10    
TRAIN_RATIO = 0.9        
TOP_K = 10         

print("=" * 80)
print("🚀 NOTEBOOK 5: RANKING & DUAL SCENARIO EVALUATION")
print("   Strategy: Hybrid (Model + Heuristic + Safety Net)")
print("=" * 80)

# =============================================================================
# [1/7] LOAD ARTIFACTS
# =============================================================================
print("\n[1/7] LOADING DATA...")

# 1. Load Candidates & Segments
try:
    with open(f"{CANDIDATE_PATH}/user_candidates.pkl", "rb") as f:
        candidates_dict = pickle.load(f)
    with open(f"{CANDIDATE_PATH}/user_segments.pkl", "rb") as f:
        segments = pickle.load(f)
        
    warm_users_list = segments['warm_users']
    cold_users_list = segments['cold_users']
    
    print(f"   ✓ Segments Loaded:")
    print(f"     - Warm Users (Model): {len(warm_users_list):,}")
    print(f"     - Cold Users (Fill):  {len(cold_users_list):,}")
    
except FileNotFoundError:
    raise FileNotFoundError("❌ Chạy Notebook 4 trước để có file candidates/segments!")

# 2. Load Ground Truth (FIXED NESTED LIST ERROR)
df_gt = pl.read_parquet(f"{FEATURE_PATH}/test_ground_truth.parquet")
if "item_id" in df_gt.columns: df_gt = df_gt.rename({"item_id": "product_id"})

# FIX: Explode trước để đảm bảo dữ liệu phẳng
df_gt = df_gt.select(["customer_id", "product_id"]).explode("product_id")

# Gom nhóm lại thành list phẳng
gt_dict = df_gt.group_by("customer_id").agg(pl.col("product_id")).to_dict(as_series=False)
ground_truth = dict(zip(gt_dict["customer_id"], gt_dict["product_id"]))

print(f"   ✓ Ground Truth Users: {len(ground_truth):,}")

# 3. Load Features
print("   -> Loading Features Tables...")
df_cust = pl.read_parquet(f"{FEATURE_PATH}/customer_features.parquet")
df_item = pl.read_parquet(f"{FEATURE_PATH}/item_features.parquet")
df_inter = pl.read_parquet(f"{FEATURE_PATH}/interaction_features.parquet")

# Standardize Columns
def fix_col_names(df):
    if "item_id" in df.columns: df = df.rename({"item_id": "product_id"})
    return df

df_cust = fix_col_names(df_cust)
df_item = fix_col_names(df_item)
df_inter = fix_col_names(df_inter)

# =============================================================================
# [2/7] PREPARE TRAINING DATA (WARM USERS ONLY)
# =============================================================================
print("\n[2/7] BUILDING TRAINING DATA (WARM USERS ONLY)...")

def build_ranker_data(user_list, is_train=True):
    data = []
    groups = []
    # Chỉ lấy những user có trong ground truth để train/val
    valid_users = [u for u in user_list if u in ground_truth and u in candidates_dict]
    
    for u in valid_users:
        cands = candidates_dict[u]
        true_items = set(ground_truth[u])
        
        pos_items = [i for i in cands if i in true_items]
        neg_items = [i for i in cands if i not in true_items]
        
        selected = []
        if is_train:
            if not pos_items: continue
            # Negative Sampling
            n_neg = min(len(neg_items), len(pos_items) * NEG_SAMPLE_RATIO)
            sampled_negs = random.sample(neg_items, n_neg)
            
            for i in pos_items: selected.append((u, i, 1))
            for i in sampled_negs: selected.append((u, i, 0))
            random.shuffle(selected)
        else:
            # Validation: Lấy hết candidate để tính metric chuẩn
            for i in cands:
                label = 1 if i in true_items else 0
                selected.append((u, i, label))
        
        if selected:
            data.extend(selected)
            groups.append(len(selected))
            
    return data, groups

# Split Train/Val trên tập Warm
train_u, val_u = train_test_split(warm_users_list, train_size=TRAIN_RATIO, random_state=42)
train_data_raw, train_groups = build_ranker_data(train_u, is_train=True)
val_data_raw, val_groups = build_ranker_data(val_u, is_train=False)

print(f"   ✓ Train Rows: {len(train_data_raw):,}")
print(f"   ✓ Val Rows:   {len(val_data_raw):,}")

# =============================================================================
# [3/7] FEATURE MERGING
# =============================================================================
print("\n[3/7] MERGING FEATURES...")

def enrich_features(raw_list):
    # Convert list -> DF
    df = pl.DataFrame(raw_list, schema=["customer_id", "product_id", "label"], orient="row")
    
    # Join Features
    df = (
        df
        .join(df_cust, on="customer_id", how="left")
        .join(df_item, on="product_id", how="left")
        .join(df_inter, on=["customer_id", "product_id"], how="left")
    )
    
    # Fill Nulls
    df = df.with_columns([
        pl.col("purchase_count").fill_null(0),
        pl.col("total_qty").fill_null(0),
        pl.col("days_since_last_buy").fill_null(999),
        
        # Derived Features
        (pl.col("price") - pl.col("avg_order_value")).abs().fill_null(0).alias("price_diff"),
        (pl.col("purchase_count") / (pl.col("transaction_count") + 1)).fill_null(0).alias("loyalty_score")
    ])
    return df

df_train = enrich_features(train_data_raw)
df_val = enrich_features(val_data_raw)

# Chọn features để train
exclude_cols = ["customer_id", "product_id", "label", "t_dat", "timestamp", "last_buy_date", 
                "registration_date", "description", "detail_desc"]
feature_cols = [c for c in df_train.columns if c not in exclude_cols and df_train[c].dtype in [pl.Float64, pl.Int64, pl.Int32, pl.Float32]]

print(f"   ✓ Model Features ({len(feature_cols)}): {feature_cols[:5]}...")

# =============================================================================
# [4/7] TRAIN MODEL (WARM USERS ONLY)
# =============================================================================
print("\n[4/7] TRAINING LIGHTGBM RANKER...")

X_train = df_train.select(feature_cols).to_pandas()
y_train = df_train["label"].to_pandas()
X_val = df_val.select(feature_cols).to_pandas()
y_val = df_val["label"].to_pandas()

del df_train, df_val, train_data_raw, val_data_raw
gc.collect()

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

ranker.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_val, y_val)],
    eval_group=[val_groups],
    eval_at=[TOP_K],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)]
)

print("   ✓ Model Trained.")
del X_train, y_train, X_val, y_val
gc.collect()

# =============================================================================
# [5/7] HYBRID INFERENCE WITH SAFETY NET (FIXED)
# =============================================================================
print("\n[5/7] GENERATING PREDICTIONS (HYBRID + SAFETY NET)...")

# 0. Load Fallback Items (Global Trends)
try:
    with open(f"{FEATURE_PATH}/cold_start_maps.pkl", "rb") as f:
        maps = pickle.load(f)
        global_fallback = maps.get("global", [])[:TOP_K]
except:
    print("   ⚠ Warning: cold_start_maps.pkl not found. Calculating from item features...")
    # Fallback nếu thiếu file: Lấy top item phổ biến nhất từ df_item
    global_fallback = df_item.sort("i_count_30d", descending=True).head(TOP_K)["product_id"].to_list()

print(f"   ✓ Global Fallback loaded: {len(global_fallback)} items")
final_preds = {}

# --- A. PREDICT FOR WARM USERS (USE MODEL) ---
print(f"   -> Phase 1: Predicting for Warm Users ({len(warm_users_list):,})...")

def batch_inference(user_list):
    results = {}
    batch_raw = []
    
    for u in user_list:
        if u in candidates_dict:
            for item in candidates_dict[u]:
                batch_raw.append((u, item, 0))
    
    if not batch_raw: return {}
    
    # Feature + Predict
    df_batch = enrich_features(batch_raw)
    scores = ranker.predict(df_batch.select(feature_cols).to_pandas())
    
    # Sort & Pick Top K
    df_batch = (
        df_batch.select(["customer_id", "product_id"])
        .with_columns(pl.Series("score", scores))
        .sort(["customer_id", "score"], descending=[False, True])
        .group_by("customer_id")
        .agg(pl.col("product_id").head(TOP_K))
    )
    
    for row in df_batch.iter_rows():
        results[str(row[0])] = row[1]
    return results

BATCH_SIZE = 10000
for i in range(0, len(warm_users_list), BATCH_SIZE):
    batch = warm_users_list[i : i+BATCH_SIZE]
    final_preds.update(batch_inference(batch))
    print(f"\r      Progress: {min(i+BATCH_SIZE, len(warm_users_list))}/{len(warm_users_list)}", end="")
print()

# --- B. FILL FOR COLD USERS (BYPASS MODEL) ---
print(f"   -> Phase 2: Filling for Cold Users ({len(cold_users_list):,})...")
count_cold_filled = 0
for u in cold_users_list:
    if u in candidates_dict:
        final_preds[str(u)] = candidates_dict[u][:TOP_K]
        count_cold_filled += 1
print(f"      Filled {count_cold_filled:,} cold users.")

# --- C. SAFETY NET (CRITICAL FIX: FILL GHOST USERS) ---
# Lấp đầy những user có trong Ground Truth nhưng chưa có dự đoán
print(f"   -> Phase 3: Safety Net (Filling gaps for Test Users)...")
missing_count = 0
for u_int in ground_truth.keys():
    u_str = str(u_int)
    if u_str not in final_preds:
        final_preds[u_str] = global_fallback
        missing_count += 1

print(f"      🔥 FILLED GAP for {missing_count:,} ghost users with Global Trends.")
print(f"   ✓ Total Predictions Ready: {len(final_preds):,}")

# =============================================================================
# [6/7] EVALUATION (DUAL SCENARIO) - FIXED
# =============================================================================
print("\n" + "="*80)
print("[6/7] FINAL METRICS REPORT")
print("="*80)

def calculate_ndcg(pred, actual, k):
    dcg = 0.0
    idcg = 0.0
    for i, p in enumerate(pred[:k]):
        if p in actual:
            dcg += 1.0 / math.log2(i + 2)
    for i in range(min(len(actual), k)):
        idcg += 1.0 / math.log2(i + 2)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_scenario(target_users_list, scenario_name):
    precisions, recalls, ndcgs = [], [], []
    
    # Xác định tập user cần đánh giá
    valid_users_raw = [u for u in target_users_list if u in ground_truth]
    
    # Nếu là FULL SYSTEM -> Đánh giá trên toàn bộ Ground Truth (không bỏ sót ai)
    if "FULL SYSTEM" in scenario_name:
        valid_users_raw = list(ground_truth.keys())
    
    print(f"📊 {scenario_name}")
    print(f"   (Evaluated on {len(valid_users_raw):,} users)")

    for u in valid_users_raw:
        actual = set(ground_truth[u])
        u_str = str(u)
        
        # Lấy dự đoán (đã được fill đầy đủ nhờ Safety Net)
        pred = final_preds.get(u_str, global_fallback)
        
        # Calculate Metrics
        hits = len(set(pred) & actual)
        precisions.append(hits / TOP_K)
        recalls.append(hits / len(actual) if actual else 0)
        ndcgs.append(calculate_ndcg(pred, actual, TOP_K))
        
    print("-" * 40)
    print(f"   👉 Precision@{TOP_K}:  {np.mean(precisions):.4%}")
    print(f"   👉 Recall@{TOP_K}:     {np.mean(recalls):.4%}")
    print(f"   👉 NDCG@{TOP_K}:       {np.mean(ndcgs):.4%}")
    print("\n")

# --- SCENARIO 1: MODEL POWER (Warm Users Only) ---
evaluate_scenario(warm_users_list, "SCENARIO 1: NO COLD START (Warm Users Only)")

# --- SCENARIO 2: REAL WORLD (Full System) ---
evaluate_scenario(segments['all_users'], "SCENARIO 2: FULL SYSTEM (All Test Users)")

# =============================================================================
# [7/7] SAVING
# =============================================================================
print("[7/7] SAVING FINAL RESULTS...")
with open(f"{OUTPUT_PATH}/predictions.json", "w") as f:
    json.dump(final_preds, f)

# Create submission csv
sub_list = []
for u, items in final_preds.items():
    sub_list.append({"customer_id": u, "prediction": " ".join(map(str, items))})
df_sub = pd.DataFrame(sub_list)
df_sub.to_csv(f"{OUTPUT_PATH}/submission.csv", index=False)

print(f"   ✓ Saved predictions.json and submission.csv to {OUTPUT_PATH}")
print("✅ NOTEBOOK 5 COMPLETED SUCCESSFULLY.")

🚀 NOTEBOOK 5: RANKING & DUAL SCENARIO EVALUATION
   Strategy: Hybrid (Model + Heuristic + Safety Net)

[1/7] LOADING DATA...
   ✓ Segments Loaded:
     - Warm Users (Model): 1,153,598
     - Cold Users (Fill):  0
   ✓ Ground Truth Users: 644,970
   -> Loading Features Tables...

[2/7] BUILDING TRAINING DATA (WARM USERS ONLY)...
   ✓ Train Rows: 7,590,517
   ✓ Val Rows:   3,061,930

[3/7] MERGING FEATURES...
   ✓ Model Features (14): ['total_revenue', 'avg_order_value', 'total_spend', 'recency_days', 'user_repurchase_ratio']...

[4/7] TRAINING LIGHTGBM RANKER...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.308238 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3453
[LightGBM] [Info] Number of data points in the train set: 7590517, number of used features: 14
Training until validation scores don't improve for 30 rounds
[50]	valid_0's n